In [ ]:
!pip install -q kailash kailash-ml polars torch torchvision matplotlib python-dotenv gymnasium


# MLFP05 — Exercise 8.4: Algorithm Comparison — Random vs DQN vs PPO


**WHAT YOU'LL LEARN:**
After completing this exercise, you will be able to:
- Compare Random, DQN, and PPO policies side-by-side on CartPole-v1
- Analyse sample efficiency: how many environment interactions does
each algorithm need to reach a given performance level?
- Measure wall-clock training time for each algorithm
- Build a decision framework: which algorithm for which business problem?
- Explain how PPO connects to RLHF for LLM alignment (bridge to M6)


**PREREQUISITES: M5/ex_8/01_dqn.py and M5/ex_8/02_ppo.py.**

**ESTIMATED TIME: ~30 min**
DATASETS: No static dataset — the environment IS the data source.
- CartPole-v1 (Gymnasium classic control, 4-D state, 2 actions)


**TASKS:**
1. Train DQN and PPO on CartPole-v1 with timing
2. Evaluate Random vs DQN vs PPO side-by-side
3. Visualise: reward comparison, sample efficiency, training time
4. Apply: decision framework for engineering managers



In [ ]:
from __future__ import annotations

import asyncio
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

import gymnasium as gym
from gymnasium import spaces

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from shared.mlfp05.ex_8 import (
    DQN,
    OUTPUT_DIR,
    ReplayBuffer,
    device,
    evaluate_policy,
    make_cartpole,
    moving_average,
    setup_engines,
)
from kailash_ml import ModelVisualizer


## TASK 1 — Train DQN and PPO with Timing
We retrain both algorithms from scratch with identical episode counts
so the comparison is fair. We measure wall-clock time for each.


In [ ]:
print("=" * 70)
print("  TASK 1: Train DQN and PPO with Timing")
print("=" * 70)

cartpole_env, obs_dim, n_actions = make_cartpole()
conn, tracker, exp_name, registry, has_registry = setup_engines()


### ActorCritic for PPO (needed for this comparison file) ────────────


In [ ]:
class ActorCritic(nn.Module):
    """Shared trunk with actor and critic heads for PPO."""

    def __init__(self, obs_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
        )
        self.policy_head = nn.Linear(hidden, n_actions)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        h = self.trunk(x)
        return self.policy_head(h), self.value_head(h).squeeze(-1)

    def act(self, state: np.ndarray) -> tuple[int, torch.Tensor, torch.Tensor]:
        s = torch.from_numpy(state.astype(np.float32)).to(device)
        logits, value = self.forward(s)
        dist = Categorical(logits=logits)
        a = dist.sample()
        return int(a.item()), dist.log_prob(a).detach(), value.detach()


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    """GAE computation for PPO."""
    advantages = [0.0] * len(rewards)
    gae = 0.0
    next_value = 0.0
    for t in reversed(range(len(rewards))):
        nonterminal = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * next_value * nonterminal - float(values[t])
        gae = delta + gamma * lam * nonterminal * gae
        advantages[t] = gae
        next_value = float(values[t])
    returns = [a + float(v) for a, v in zip(advantages, values)]
    return advantages, returns


### Train DQN ────────────────────────────────────────────────────────


In [ ]:
N_DQN_EPISODES = 200
N_DQN_ENV_STEPS = 0  # count total environment interactions


async def _train_dqn_timed():
    global N_DQN_ENV_STEPS
    q_net = DQN(obs_dim, n_actions).to(device)
    target_net = DQN(obs_dim, n_actions).to(device)
    target_net.load_state_dict(q_net.state_dict())
    target_net.eval()
    optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
    replay = ReplayBuffer(capacity=10_000)
    epsilon = 1.0
    episode_rewards: list[float] = []
    env_steps = 0

    async with tracker.run(experiment_name=exp_name, run_name="comparison_dqn") as ctx:
        await ctx.log_params({"algorithm": "DQN", "episodes": str(N_DQN_EPISODES)})
        for ep in range(N_DQN_EPISODES):
            state, _ = cartpole_env.reset(seed=42 + ep)
            total_reward = 0.0
            done = False
            while not done:
                # TODO: Epsilon-greedy action selection
                # Hint: same pattern as 01_dqn.py
                if random.random() < epsilon:
                    action = # TODO
                else:
                    with torch.no_grad():
                        s_t = torch.tensor(state, dtype=torch.float32, device=device)
                        action = # TODO
                next_state, reward, terminated, truncated, _ = cartpole_env.step(action)
                done = terminated or truncated
                replay.push(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward
                env_steps += 1
                # TODO: DQN training step when replay has enough samples
                # Hint: same pattern as 01_dqn.py — sample, Q-values, targets, MSE loss
                if len(replay) >= 500:
                    s_b, a_b, r_b, ns_b, d_b = replay.sample(64)
                    q_values = # TODO
                    with torch.no_grad():
                        next_q = # TODO
                        targets = # TODO
                    loss = # TODO
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
            epsilon = max(0.01, epsilon * 0.995)
            if (ep + 1) % 10 == 0:
                target_net.load_state_dict(q_net.state_dict())
            episode_rewards.append(total_reward)
            await ctx.log_metric("episode_reward", total_reward, step=ep)
    N_DQN_ENV_STEPS = env_steps
    return q_net, episode_rewards


print("\n  Training DQN (200 episodes)...")
dqn_start = time.time()
dqn_model, dqn_rewards = await _train_dqn_timed()
dqn_time = time.time() - dqn_start
print(
    f"  DQN: {dqn_time:.1f}s, {N_DQN_ENV_STEPS} env steps, final avg20={np.mean(dqn_rewards[-20:]):.1f}"
)


### Train PPO ────────────────────────────────────────────────────────


In [ ]:
N_PPO_ITERS = 30
STEPS_PER_ITER = 1024
N_PPO_ENV_STEPS = N_PPO_ITERS * STEPS_PER_ITER


async def _train_ppo_timed():
    model = ActorCritic(obs_dim, n_actions).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    iter_returns: list[float] = []

    async with tracker.run(experiment_name=exp_name, run_name="comparison_ppo") as ctx:
        await ctx.log_params({"algorithm": "PPO", "iterations": str(N_PPO_ITERS)})
        for it in range(N_PPO_ITERS):
            # Collect trajectory
            states, actions, log_probs, values, rewards, dones = [], [], [], [], [], []
            state, _ = cartpole_env.reset(seed=int(np.random.randint(0, 100_000)))
            for _ in range(STEPS_PER_ITER):
                action, log_prob, value = model.act(state)
                next_state, reward, terminated, truncated, _ = cartpole_env.step(action)
                states.append(state.astype(np.float32))
                actions.append(action)
                log_probs.append(log_prob)
                values.append(value)
                rewards.append(float(reward))
                done = terminated or truncated
                dones.append(done)
                state = next_state
                if done:
                    state, _ = cartpole_env.reset(
                        seed=int(np.random.randint(0, 100_000))
                    )

            advantages, returns = compute_gae(rewards, values, dones)
            s_t = torch.tensor(np.stack(states), dtype=torch.float32, device=device)
            a_t = torch.tensor(actions, dtype=torch.long, device=device)
            old_lp_t = torch.stack(log_probs).to(device)
            adv_t = torch.tensor(advantages, dtype=torch.float32, device=device)
            ret_t = torch.tensor(returns, dtype=torch.float32, device=device)
            adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)

            n = s_t.size(0)
            idxs = np.arange(n)
            # TODO: PPO update — multiple epochs over minibatches
            # Hint: same clipped surrogate pattern as 02_ppo.py
            for _ in range(4):
                np.random.shuffle(idxs)
                for start in range(0, n, 256):
                    mb = idxs[start : start + 256]
                    logits, vpred = model(s_t[mb])
                    dist = Categorical(logits=logits)
                    new_lp = dist.log_prob(a_t[mb])
                    ratio = # TODO: torch.exp(new_lp - old_lp_t[mb])
                    surr1 = # TODO
                    surr2 = # TODO: torch.clamp(ratio, 0.8, 1.2) * adv_t[mb]
                    policy_loss = # TODO: -torch.min(surr1, surr2).mean()
                    value_loss = # TODO: F.mse_loss(vpred, ret_t[mb])
                    entropy = # TODO: dist.entropy().mean()
                    loss = policy_loss + 0.5 * value_loss - 0.01 * entropy
                    opt.zero_grad()
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                    opt.step()

            ep_rs: list[float] = []
            running = 0.0
            for r, d in zip(rewards, dones):
                running += r
                if d:
                    ep_rs.append(running)
                    running = 0.0
            avg_ret = float(np.mean(ep_rs)) if ep_rs else float(running)
            iter_returns.append(avg_ret)
            await ctx.log_metric("avg_episode_return", avg_ret, step=it)
    return model, iter_returns


print("  Training PPO (30 iterations x 1024 steps)...")
ppo_start = time.time()
ppo_model, ppo_returns = await _train_ppo_timed()
ppo_time = time.time() - ppo_start
print(
    f"  PPO: {ppo_time:.1f}s, {N_PPO_ENV_STEPS} env steps, final return={ppo_returns[-1]:.1f}"
)


### Checkpoint 1 ─────────────────────────────────────────────────────


In [ ]:
assert len(dqn_rewards) == N_DQN_EPISODES
assert len(ppo_returns) == N_PPO_ITERS
print("--- Checkpoint 1 passed --- both algorithms trained with timing\n")


## TASK 2 — Evaluate Random vs DQN vs PPO side-by-side


In [ ]:
print("=" * 70)
print("  TASK 2: Evaluate Random vs DQN vs PPO")
print("=" * 70)


def random_policy(state):
    return cartpole_env.action_space.sample()


def dqn_policy(state):
    with torch.no_grad():
        s = torch.tensor(state, dtype=torch.float32, device=device)
        return int(dqn_model(s).argmax().item())


def ppo_policy(state):
    with torch.no_grad():
        s = torch.from_numpy(state.astype(np.float32)).to(device)
        logits, _ = ppo_model(s)
        return int(logits.argmax().item())


N_EVAL = 50  # more episodes for statistical significance
random_returns = evaluate_policy(cartpole_env, random_policy, n_episodes=N_EVAL)
dqn_eval_returns = evaluate_policy(cartpole_env, dqn_policy, n_episodes=N_EVAL)
ppo_eval_returns = evaluate_policy(cartpole_env, ppo_policy, n_episodes=N_EVAL)

print(f"\n  Policy Comparison (CartPole-v1, {N_EVAL} eval episodes)")
print(f"  {'Policy':<10} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8} {'Median':>8}")
print(f"  {'-'*50}")
for name, returns in [
    ("Random", random_returns),
    ("DQN", dqn_eval_returns),
    ("PPO", ppo_eval_returns),
]:
    print(
        f"  {name:<10} {np.mean(returns):>8.1f} {np.std(returns):>8.1f} "
        f"{np.min(returns):>8.1f} {np.max(returns):>8.1f} {np.median(returns):>8.1f}"
    )


### Checkpoint 2 ─────────────────────────────────────────────────────


In [ ]:
assert float(np.mean(dqn_eval_returns)) > float(
    np.mean(random_returns)
), "DQN should outperform random"
assert float(np.mean(ppo_eval_returns)) > float(
    np.mean(random_returns)
), "PPO should outperform random"
print("--- Checkpoint 2 passed --- both algorithms outperform random\n")


## TASK 3 — Visualise: reward comparison, sample efficiency, training time


In [ ]:
print("=" * 70)
print("  TASK 3: Comprehensive Comparison Visualisations")
print("=" * 70)

viz = ModelVisualizer()


### Plot 1: Evaluation reward box plot ───────────────────────────────


In [ ]:
# TODO: Create box plot comparing Random, DQN, PPO evaluation returns
# Hint: pl.DataFrame with "Policy" and "Evaluation Return" columns
comparison_df = # TODO
fig1 = # TODO: viz.box_plot(...)
fig1.write_html(str(OUTPUT_DIR / "04_policy_comparison_boxplot.html"))
print(f"  Saved: {OUTPUT_DIR / '04_policy_comparison_boxplot.html'}")
# INTERPRETATION: The box plot shows final policy quality. Random is
# clustered around ~20 (the pole falls quickly). DQN and PPO should be
# much higher and with lower variance — they've learned stable policies.


### Plot 2: Training curves on a common x-axis (env steps) ──────────


In [ ]:
# Normalise both algorithms to environment interactions for fair comparison
dqn_cumulative_steps = np.cumsum(
    [max(10, r) for r in dqn_rewards]
).tolist()
ppo_cumulative_steps = [(i + 1) * STEPS_PER_ITER for i in range(len(ppo_returns))]

# TODO: Create a line plot with DQN and PPO training curves on env-steps x-axis
# Hint: go.Figure() with two go.Scatter traces + random baseline hline
fig2 = # TODO
fig2.write_html(str(OUTPUT_DIR / "04_sample_efficiency.html"))
print(f"  Saved: {OUTPUT_DIR / '04_sample_efficiency.html'}")
# INTERPRETATION: Sample efficiency measures how many environment
# interactions are needed to reach a given performance level. PPO
# uses more steps per iteration (1024 batch) but often reaches good
# performance with fewer total iterations. DQN learns from replay
# (data-efficient) but takes more episodes to converge.


### Plot 3: Wall-clock training time comparison ──────────────────────


In [ ]:
# TODO: Create bar chart comparing DQN and PPO training times
# Hint: go.Figure(data=[go.Bar(x=["DQN", "PPO"], y=[dqn_time, ppo_time], ...)])
fig3 = # TODO
fig3.write_html(str(OUTPUT_DIR / "04_training_time.html"))
print(f"  Saved: {OUTPUT_DIR / '04_training_time.html'}")


### Plot 4: Summary dashboard ────────────────────────────────────────


In [ ]:
# TODO: Create a 2x2 dashboard with make_subplots
# Subplot (1,1): bar chart of final policy quality (mean +/- std)
# Subplot (1,2): training curves (DQN + PPO scatter)
# Subplot (2,1): training time bars
# Subplot (2,2): algorithm properties table
# Hint: make_subplots(rows=2, cols=2, specs=[[{"type":"bar"},{"type":"scatter"}],
#   [{"type":"bar"},{"type":"table"}]])
fig4 = # TODO
fig4.write_html(str(OUTPUT_DIR / "04_comparison_dashboard.html"))
print(f"  Saved: {OUTPUT_DIR / '04_comparison_dashboard.html'}")


### Checkpoint 3 ─────────────────────────────────────────────────────


In [ ]:
assert Path(OUTPUT_DIR / "04_policy_comparison_boxplot.html").exists()
assert Path(OUTPUT_DIR / "04_sample_efficiency.html").exists()
assert Path(OUTPUT_DIR / "04_training_time.html").exists()
assert Path(OUTPUT_DIR / "04_comparison_dashboard.html").exists()
print("--- Checkpoint 3 passed --- all comparison visualisations generated\n")


## TASK 4 — Apply: Decision Framework for Engineering Managers


In [ ]:
print("=" * 70)
print("  TASK 4: Which Algorithm for Which Business Problem?")
print("=" * 70)

# Build the decision framework as a polars DataFrame
decision_framework = pl.DataFrame(
    {
        "Business Problem": [
            "Inventory reorder (FairPrice)",
            "Surge pricing (Grab)",
            "Customer churn (Singtel)",
            "Portfolio rebalancing",
            "Queue staffing (Changi)",
            "Traffic signals (LTA)",
            "Energy trading (SP Group)",
            "LLM alignment (M6)",
        ],
        "Action Space": [
            "Discrete (4 order sizes)",
            "Continuous (price multiplier)",
            "Discrete (4 interventions)",
            "Multi-discrete (27 combos)",
            "Discrete (7 shifts)",
            "Discrete (5 allocations)",
            "Discrete (5 trade sizes)",
            "Continuous (token probs)",
        ],
        "Recommended": [
            "DQN",
            "PPO",
            "DQN",
            "PPO",
            "DQN",
            "DQN",
            "DQN or PPO",
            "PPO (RLHF)",
        ],
        "Why": [
            "Small discrete space, clear reward signal, replay buffer helps with sparse reorders",
            "Continuous prices need policy gradients; PPO handles smoothly",
            "Small discrete space, DQN learns value of each intervention",
            "Large action space (27); PPO scales better than DQN",
            "Small discrete space, DQN works well",
            "Small discrete space, fast convergence needed",
            "Either works; PPO if extending to continuous trade sizes",
            "PPO is standard for RLHF — directly optimises token policy",
        ],
    }
)

print("\n  ALGORITHM SELECTION GUIDE")
print("  " + "=" * 76)
for row in decision_framework.iter_rows(named=True):
    print(f"\n  {row['Business Problem']}")
    print(f"    Action space: {row['Action Space']}")
    print(f"    Recommended:  {row['Recommended']}")
    print(f"    Why: {row['Why']}")


### Summary statistics ───────────────────────────────────────────────


In [ ]:
print("\n\n  TRAINING SUMMARY")
print("  " + "=" * 60)
print(f"  {'Metric':<30} {'DQN':>12} {'PPO':>12}")
print(f"  {'-'*54}")
print(f"  {'Training episodes/iters':<30} {N_DQN_EPISODES:>12} {N_PPO_ITERS:>12}")
print(f"  {'Total env interactions':<30} {N_DQN_ENV_STEPS:>12} {N_PPO_ENV_STEPS:>12}")
print(f"  {'Wall-clock time (s)':<30} {dqn_time:>12.1f} {ppo_time:>12.1f}")
print(
    f"  {'Eval mean reward':<30} {np.mean(dqn_eval_returns):>12.1f} {np.mean(ppo_eval_returns):>12.1f}"
)
print(
    f"  {'Eval std reward':<30} {np.std(dqn_eval_returns):>12.1f} {np.std(ppo_eval_returns):>12.1f}"
)
print(f"  {'Uses replay buffer':<30} {'Yes':>12} {'No':>12}")
print(f"  {'On/off policy':<30} {'Off-policy':>12} {'On-policy':>12}")
print(f"  {'Continuous actions':<30} {'No':>12} {'Yes':>12}")


### Checkpoint 4 ─────────────────────────────────────────────────────


In [ ]:
assert len(decision_framework) == 8, "Decision framework should cover 8 problems"
print("\n--- Checkpoint 4 passed --- decision framework complete\n")

cartpole_env.close()

# Clean up
await conn.close()


## REFLECTION


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — Algorithm Comparison")
print("=" * 70)
print(
    """
  [x] Compared Random vs DQN vs PPO on identical CartPole-v1 evaluation
  [x] Measured sample efficiency (env interactions to reach performance)
  [x] Measured wall-clock training time for each algorithm
  [x] Built a decision framework for engineering managers:

      USE DQN WHEN:
        - Small, discrete action space (< 10 actions)
        - Clear, immediate reward signal
        - Data efficiency matters (replay buffer reuses data)
        - Examples: inventory ordering, churn interventions, queue staffing

      USE PPO WHEN:
        - Continuous or large discrete action space
        - Stability matters more than data efficiency
        - You need on-policy guarantees (fresh data each iteration)
        - Examples: pricing, portfolio management, LLM alignment (RLHF)

      USE NEITHER (yet) WHEN:
        - You don't have a good simulator/environment
        - The reward function is unclear or hard to specify
        - Supervised learning can solve the problem (simpler, cheaper)

  BRIDGE TO M6 (RLHF — Reinforcement Learning from Human Feedback):
  Everything you've learned here IS the foundation for RLHF:
    - PPO (this exercise) = the optimisation algorithm
    - Reward model (M6) = trained on human preference rankings
    - Policy (M6) = the language model's next-token distribution
    - DPO (M6) = a shortcut that skips the reward model entirely

  The core loop is identical:
    1. Agent (LLM) takes action (generates text)
    2. Environment (human/reward model) provides reward
    3. PPO updates the policy to maximise expected reward
    4. Clipping prevents catastrophic forgetting of language ability

  You now understand RL from first principles. M6 applies it to
  language models — the only difference is the environment.
"""
)
